**1.write a pyspark query to find duplicates using window function**

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [0]:
# Sample data
data = [
    (101, "John",  "IT",10000,"2013-11-22"),
    (102, "Alice", "HR",121252,"2016-12-27"),
    (101, "John",  "IT",2000,"1997-1-22"),
    (103, "Bob",   "Finance",61235,"1998-02-16"),
    (102, "Alice", "HR",700,"2025-12-01")
]

# Column names
columns = ["emp_id", "name", "dept","salary","hire_date"]

# Create DataFrame
df4 = spark.createDataFrame(data, columns)

# Display DataFrame
df4.show()

df4.createOrReplaceTempView("employees")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

window_spec = Window.partitionBy("emp_id", "name", "dept").orderBy( "emp_id")

df_with_rn = df4.withColumn("rn",row_number().over(window_spec))
df_with_rn.show()
# Duplicate rows
duplicates_df = df_with_rn.filter("rn > 1")
duplicates_df.show()

In [0]:
#Remove Duplicates (Keep First Record)
dedup_df = df_with_rn.filter("rn = 1")
dedup_df.show()


**Find Duplicate Rows Using SQL + Window Function**

In [0]:
df1=df4.createOrReplaceTempView("employees")

In [0]:
%sql
WITH cte AS (
    SELECT *,
    row_number() OVER (PARTITION BY emp_id, name, dept ORDER BY emp_id ) AS rn
    FROM employees
)
SELECT *
FROM cte
WHERE rn > 1

### 2.How do you find the second highest salary from an `employees` table?

QUALIFY is a non-standard SQL extension. It is fully supported in modern cloud data platforms like Snowflake, Google BigQuery, Databricks, and Amazon Redshift.
In a SELECT statement, the QUALIFY clause filters the results of window functions.

In [0]:
%sql
/* with cte as
(
  select 
        emp_id,salary,name,
        dense_rank() over(order by salary DESC)as second_sal
  from employees
)
select *
from cte
where second_sal = 2 */

-- using qualify keyword--
select emp_id,salary,name,
       dense_rank() over(order by salary DESC)as second_sal
from employees
qualify second_sal=2

from pyspark.sql.functions import col
- > col() in PySpark is used to create a **Column object** that represents a DataFrame column. It comes from pyspark.sql.functions.
- > Many PySpark functions expect Column objects, not strings.

In [0]:
#second highest salary
from pyspark.sql.functions import *
from pyspark.sql.window import Window
window_spec= Window.orderBy(col("salary").desc())
second_saldf=df4.withColumn("second_sal", dense_rank().over(window_spec)).select("*")
second_saldf1=second_saldf.filter(col("second_sal")==2)
display(second_saldf1)

### How do you identify duplicate records in a table (e.g., `Cars1`) based on the `name` column?

In [0]:
%sql
select emp_id,name,
       row_number() over(partition by name order by emp_id) as dup_names
from employees
qualify dup_names>1

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
window_spec=Window.partitionBy(col("name")).orderBy(col("emp_id"))
df5=df4.withColumn("rn", row_number().over(window_spec)).filter(col("rn")>1)
display(df5)

### What is the most performance-conscious way to fetch exactly the first chronological record from an `employees` table sorted by `hire_date`?

In [0]:
%sql
select *
from employees
order by hire_date

In [0]:
display(df4.orderBy(col("hire_date")))


| Function              | Returns                               |
| --------------------- | ------------------------------------- |
| `current_date()`      | `2026-06-23` (DateType)               |
| `current_timestamp()` | `2026-06-23 14:35:22` (TimestampType) |


### How do you write a unified query to retrieve both the top 2 and bottom 2 records based on salary in a single result set?

In [0]:
%sql
(select *
from employees
order by salary desc
limit 2)
union
(select *
from employees
order by salary
limit 2)

In [0]:
df1=(df4.orderBy(col("salary").desc()).limit(2))
df2 = ((df4.orderBy(col("salary")).limit(2)))
df3=display(df1.union(df2))


### How do you extract the 3rd highest distinct salary from the `employees` table in Databricks and BigQuery?

In [0]:
%sql
select *,
     dense_rank() over(order by salary desc) as drank
from employees
qualify drank=3

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
window_spec=Window.orderBy(col("salary").desc())
df5=df4.withColumn("dr",dense_rank().over(window_spec)).filter(col("dr")==3).select("*")
display(df5)

### How do you filter only odd-numbered records (1st, 3rd, 5th, etc.) from an ordered employee dataset in a cloud warehouse?

In [0]:
%sql
---even---
select *
from employees
where salary%2=0;



In [0]:
%sql
---odd---
select *
From employees
where salary%2<>0

### How do you create a new empty table schema copied from `employees` without copying any data in Databricks?

In [0]:
%sql
--CTAS---where 1 =0 only schema copy---
CREATE TABLE IF NOT EXISTS new_table_employees
AS SELECT *
FROM employees
WHERE 1 = 0; 

/*select *
from new_table_employees */

In [0]:
df = spark.table("employees")
#above code equivalent to df = spark.sql("SELECT * FROM employees")
df.limit(0).write.mode("overwrite").saveAsTable("new_table1")
display("new_table")

**PERCENT_RANK()** is a window function that calculates the relative rank of a row within a partition and returns a value between 0 and 1

-> Top 10% Employees
-> WHERE pct_rank <= 0.10;

### How do you retrieve the bottom 50% of records from the `employees` table in BigQuery, Synapse, and Databricks?

In [0]:
%sql
select *,
       percent_rank() over(order by salary) as pr
from employees
qualify pr > 0.5


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *

window_spec=Window.orderBy(col("salary"))
df = df4.withColumn("pr",percent_rank().over(window_spec)).filter(col("pr")>0.5)
display(df)

### What is the standard DDL pattern to duplicate both the structure and data of an `employees` table in a cloud data warehouse?

In [0]:
%sql
--CTAS--
create table if not exists copy_data_schema as
select * from employees; 


In [0]:
df = spark.table("employees")
#display(df)
df.write.mode("overwrite").saveAsTable("new_employees")
display(spark.table("new_employees"))

### How do you retrieve exactly matching records present in both `employees` and `employees_staging` using standard set operators?

Both DataFrames must have the same schema (same number of columns and compatible data types).
intersect() removes duplicate rows in the result
> df1.intersect(df2).show()

In [0]:
%skip
%sql

SELECT * FROM employees
INTERSECT
SELECT * FROM employees_staging;


In [0]:
%skip
df=(df4.select("*")).intersect(df2.select("*"))

### What query identifies employees mapped to a `department_id` that does not exist in the `departments` table?

In [0]:
from pyspark.sql.functions import *
columns = ["name","id","department"]
data=[("sachin","123","ABC"),
       ("tabla","345","cab")]
df=spark.createDataFrame(data,columns)
display(df)

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
data = [
    (101, "Alice",   "Mumbai",    "2022-01-10", "2025-06-01 10:00:00"),
    (102, "Bob",     "Delhi",     "2021-03-15", "2025-06-02 11:10:00"),
    (103, "Charlie", "Pune",      "2020-07-22", "2025-06-01 09:00:00"),
    (104, "David",   None,        "2019-11-01", "2025-06-03 14:25:00"),
    (105, "Eva",     "Mumbai",    "2023-02-05", "2025-06-04 08:45:00"),
    (106, None,      "Bangalore", "2022-09-12", "2025-06-05 12:00:00"),  # null name
    (107, "Grace",   "Chennai",   "2018-04-18", "2025-06-06 13:30:00"),
    (101, "Alice",   "Mumbai",    "2022-01-10", "2026-06-24 4:26:00"),
    (102, "Bob",     "Delhi",     "2021-03-15", "2026-06-23 11:10:00"),
    (108, "Henry",   "Hyderabad", "2021-12-01", "2025-06-07 16:10:00")
]
columns = ["customer_id", "customer_name", "city", "signup_date", "updated_at"]
customers = spark.createDataFrame(data,columns)
customers.createOrReplaceTempView("customers45")
customers.show()


In [0]:
data = [
    (1001, 101, "2025-06-01", "2025-06-03", "2025-06-03 10:00:00", 2500, "DELIVERED"),
    (1002, 101, "2025-06-05", None,        "2025-06-05 12:00:00", 1800, "CANCELLED"),
    
    (1003, 102, "2025-06-02", "2025-06-04", "2025-06-04 14:20:00", 3000, "DELIVERED"),
    (1003, 102, "2025-06-02", "2025-06-04", "2025-06-04 14:20:00", 3005, "DELIVERED"),  # duplicate
    
    (1004, 105, "2025-06-03", "2025-06-06", "2025-06-06 09:15:00", 1500, "DELIVERED"),
    
    (1005, 108, "2025-06-04", None,        "2025-06-04 18:00:00", None,  "PENDING"),
    
    (1006, 107, "2025-06-05", "2025-06-07", "2025-06-07 11:30:00", 5000, "DELIVERED"),
    
    (1007, 999, "2025-06-08", "2025-06-09", "2025-06-09 10:00:00", 2000, "DELIVERED")  # orphan order
]
columns = [
    "order_id",
    "customer_id",
    "order_placed_date",
    "delivered_date",
    "updated_at",
    "amount",
    "status"
]
orders = spark.createDataFrame(data, columns)
orders.createOrReplaceTempView("orders45")
orders.show()

### What query identifies customers mapped to a `orders` that does not exist in the `orders` table?
> Customers Who never placed any order

In [0]:
%sql
select c.customer_id,customer_name,order_id,amount,status
from customers45 as c
Left join orders45 as o
on c.customer_id = o.customer_id
where o.order_id is null

In [0]:
df_leftjoin=customers.join(orders,"customer_id","left").filter(col("order_id").isNull()).select("customer_id","customer_name","city","order_id","status")
display(df_leftjoin)

### How do you extract unique records from an `employees` dataset without using the DISTINCT keyword?

In [0]:
%sql
/* select distinct name
from employees */

/*select name,emp_id,
      count(*) as cnt
from employees
Group by name, emp_id */



In [0]:
from pyspark.sql.functions import *
df2=df4.groupBy("name","emp_id").agg(count("*").alias("cnt")).select("name", "emp_id", "cnt")
display(df2)

### What is the most performant standard syntax to query employees whose first names are exactly 'John' or 'Bob'?

In [0]:
%sql
select *
from employees
where name IN ("John","Bob")

In [0]:
%sql
select *
from employees
where name like "J%"

In [0]:
df5=df4.filter(col("name").isin("John","Bob")).show()

### In Google BigQuery, how do you filter employees who were hired in the year 2013 from a `DATETIME` or `DATE` column?

In [0]:
%sql
select *
from employees
where YEAR(hire_date)='2013'

In [0]:
df4.filter(year(col("hire_date"))=="2013").show()

### What is the correct standard aggregation query to find the maximum salary in each department?

In [0]:
%sql
select dept,
       max(salary) as max_sal
from employees
Group by dept

In [0]:
df4.groupBy("dept").agg(max("salary").alias("max_sal")).show()

In [0]:
%sql
select *,
       row_number() over(partition by dept order by salary desc) as highest_sal 
from employees
qualify highest_sal = 1

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
window_spec = Window.partitionBy("dept").orderBy(col("salary").desc())
df1=df4.withColumn("high_sal",row_number().over(window_spec)).filter(col("high_sal")==1)
display(df1)

### How do you query employees hired in 2007 who make a salary greater than 1000 in BigQuery?

In [0]:
df4.filter((year(col("hire_date"))==2013) & (col("salary")>1000)).show()

In [0]:
%sql
select *
from employees
where year(hire_date) = "2013" and salary >"1000"

### How do you dynamically generate rows containing numbers 1 to 100 without querying a physical table in BigQuery?

In [0]:
%sql
select sequence(1,100)

### In Databricks (using Delta Lake), how do you remove duplicate rows from a table, keeping only the record with the oldest insertion timestamp?

In [0]:
%sql
select *
from customers45

In [0]:
%sql
select *
from customers45

In [0]:
%skip
%sql
/* with dup_cte as
(
   select *,
         row_number() over(partition by customer_id,customer_name order by updated_at asc) as rn
   From customers45
   where customer_id is not null
)
delete from dup_cte 
where rn=1 */

--for delta tables --
MERGE INTO customers45 AS t
USING (
  SELECT 
    customer_id, 
    MIN(updated_at) as old_updated_at
  FROM customers45
  GROUP BY customer_id
) AS s
ON t.customer_id = s.customer_id
WHEN MATCHED AND t.updated_at > s.old_updated_at THEN 
  DELETE;

---how do you remove duplicate rows from a table, keeping only the record with the oldest insertion timestamp?---
- -> t.updated_at (Latest) > s.old_updated_at (Old) (----->) Evaluates to TRUE (----->) Row is DELETED.
- -> t.updated_at (Oldest) > s.old_updated_at (Old) (----->) Evaluates to FALSE (----->) Row is KEPT.

In [0]:
%skip
%sql
WITH oldest_per_customer AS (
  SELECT customer_id, MIN(updated_at) AS old_updated_at
  FROM customers45
  GROUP BY customer_id
)
DELETE FROM customers45 AS t
WHERE EXISTS (
  SELECT 1
  FROM oldest_per_customer AS s
  WHERE s.customer_id = t.customer_id
    AND t.updated_at > s.old_updated_at
);

### 21.What query identifies and counts the frequency of duplicate keys in a table?

In [0]:
%sql

SELECT
    customer_id,
    COUNT(*) AS frequency
FROM customers45
GROUP BY customer_id
HAVING COUNT(*) > 1;


In [0]:
df5=customers.groupBy("customer_id").agg(count(col("*")).alias("cnt")).filter(col("cnt")>1)
df5.show()

### 22.How do you find the Top 2 highest-spending customers for each city?

In [0]:
customers.show()
orders.show()


In [0]:
from pyspark.sql.window import Window
join_df=customers.join(orders,"customer_id","inner")
display(join_df)
window_spec=Window.partitionBy("city").orderBy(col("amount").desc())
df2=join_df.withColumn("rnk",dense_rank().over(window_spec)).filter(col("rnk")==1)
display(df2)

In [0]:
%sql
select city,c.customer_id,c.customer_name,o.amount,
   dense_rank() over(partition by city order by o.amount desc) as rnk
from customers45 as c
Inner join orders45 as o
on c.customer_id=o.customer_id
qualify rnk=1

### 23.How do you calculate total revenue grouped by City for the transaction year 2025 where order amount >1000?

In [0]:
%sql
with join_cte as(
select *
from customers45 as c
inner join orders45 as o
on c.customer_id=o.customer_id
where year(o.order_placed_date)=2025 )
select *
from join_cte

